# FMSE Lab 12 - Evaluation Harness

**Course:** FMSE 2026 · **Module 12:** Evaluation Engineering and Verification/Validation · **Phase:** Verify & Validate · **Tier:** Senior

Course home and progress: [agentic-ai.es/academy/fmse/learn/m12-evaluation-vv](https://agentic-ai.es/academy/fmse/learn/m12-evaluation-vv).

## 0. Mission and competency target

**Mission.** Create an evaluation suite that detects at least 90% of planted faults across correctness, robustness, schema, and injection-resilience categories.

**Guided lab.** Students build scorers over a provided claims-extraction system and then add metamorphic transformations.

**Competency target (Module 12 outcomes):**
- Design golden datasets from requirements and real failure modes.
- Build deterministic, semantic, model-graded, and custom scorers with calibrated use.
- Use metamorphic testing to reveal brittleness without requiring one exact output.
- Distinguish verification (meets specified requirements) from validation (meets stakeholder intent in realistic use).

**Scaffolding:** about 25% guided, 75% independent. This is an **engineering challenge**: the guided part gives you a working reference path; the challenge does not.

## 1. Requirements and acceptance criteria

Public validators check these requirements. They report which requirement failed and why — never the expected implementation.

| ID | Requirement |
| --- | --- |
| EVL-01 | Golden set categorized |
| EVL-02 | At least 3 scorer types |
| EVL-03 | Metamorphic transformations implemented |
| EVL-04 | Verification traceability present |
| EVL-05 | Validation scenario defined |
| EVL-06 | Fault-detection target met **(critical)** |

**Competency gate (portal):** Fault-detection target met; no high-severity regression. Quiz >= 80%, the guided lab, the artifact and your reflection are also required. A critical requirement cannot be offset by other scores.

## 2. Environment setup

In [ ]:
# Idempotent: safe to re-run, and it never prints secrets.
LAB_ID = "lab-12"
LABKIT_VERSION = "1.1.0"
LABKIT_SOURCES = [
    "https://agentic-ai.es/academy/fmse/labs",  # agentic-ai.es mirror
    "https://raw.githubusercontent.com/farithjhg/fmse-course-labs/main",  # GitHub (public repositories only)
]
import hashlib, json, os, pathlib, sys, urllib.request


def _fmse_bootstrap():
    """Make fmse_labkit importable: installed copy, local checkout, cached download, or verified download."""
    try:
        import fmse_labkit
        if fmse_labkit.__version__ == LABKIT_VERSION:
            return "fmse_labkit already loaded"
    except ImportError:
        pass
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "fmse_labkit" / "__init__.py").exists():
            sys.path.insert(0, str(folder))
            return f"using the labkit in {folder}"
    target = pathlib.Path("fmse_labkit_dist").resolve()
    marker = target / f".labkit-{LABKIT_VERSION}"
    if marker.exists():
        sys.path.insert(0, str(target))
        return "using the previously downloaded labkit"
    problems = []
    for base in [os.environ.get("FMSE_LABKIT_URL")] + LABKIT_SOURCES:
        if not base:
            continue
        base = base.rstrip("/")
        try:
            manifest = json.loads(urllib.request.urlopen(base + "/manifest.json", timeout=20).read())
            if manifest.get("labkit_version") != LABKIT_VERSION:
                raise RuntimeError(f"mirror serves labkit {manifest.get('labkit_version')}")
            for entry in manifest["files"]:
                if not entry["path"].startswith("fmse_labkit/"):
                    continue
                data = urllib.request.urlopen(f"{base}/{entry['path']}", timeout=20).read()
                if hashlib.sha256(data).hexdigest() != entry["sha256"]:
                    raise RuntimeError(f"integrity check failed for {entry['path']}")
                dest = target / entry["path"]
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(data)
            marker.write_text("ok")
            sys.path.insert(0, str(target))
            return f"downloaded and verified the labkit from {base}"
        except Exception as exc:  # try the next source
            problems.append(f"{base} ({type(exc).__name__})")
    raise RuntimeError(
        "Could not load fmse_labkit from: " + "; ".join(problems) + ". "
        "Recovery: download the labs folder from agentic-ai.es/academy/fmse/resources, upload the fmse_labkit folder next to this notebook (Files panel), and re-run this cell."
    )


print(_fmse_bootstrap())
import fmse_labkit as fmse

fmse.set_language("en")
print(f"fmse_labkit {fmse.__version__} ready for {LAB_ID} ({fmse.title(fmse.lab_spec(LAB_ID))})")

## 3. Secrets check

This lab runs fully offline; no provider key is needed. Keys you use elsewhere belong in Colab Secrets and are never printed.

In [ ]:
fmse.secrets_check()

## 4. Guided experiment

A correct claims-extraction system (`reference_system`) and its contract. Build scorers against it, then check that simple transformations leave the correct output unchanged.

In [ ]:
from fmse_labkit.labs import lab12

guided = fmse.GuidedLog(LAB_ID, required_steps=["scorers_built", "metamorphic_checked"])
for email in lab12.SAMPLE_EMAILS:
    print(repr(email[:60]), "->", lab12.reference_system(email))
print("Requirements:", lab12.CLAIM_REQUIREMENTS)

## 5. Predict before you run

Will an exact-match scorer and a schema scorer agree on an output whose amount is the string '1250.50'? Write your prediction, then run the experiment.

In [ ]:
my_prediction = ""
guided.predict("prediction", my_prediction)

In [ ]:
out = lab12.reference_system(lab12.SAMPLE_EMAILS[0])
stringy = [{**c, "amount": f"{c['amount']:.2f}"} for c in out]
print("exact match:", stringy == out, "| schema errors:", fmse.schema.validate(stringy[0], lab12.CLAIM_SCHEMA))
guided.step("scorers_built")
reordered = "\n".join(reversed(lab12.SAMPLE_EMAILS[0].splitlines()))
print("invariant under reordering:", lab12.reference_system(reordered) == out)
guided.step("metamorphic_checked")
guided.outcome("prediction", "see output above")

**Worksheet.** At least one full sentence per field; these observations are guided-lab evidence.

In [ ]:
worksheet = {
    "scorer_disagreement": "",
    "metamorphic_value": "",
    "prediction_check": "",
}
guided.observe(worksheet)

## 6. Challenge

**Fault-detecting evaluation suite.** Create an evaluation suite that detects at least 90% of planted faults across correctness, robustness, schema, and injection-resilience categories.

Constraints:
- No false alarm on the correct system.
- Fault detection >= 90% of planted mutants (critical).

In [ ]:
golden_set = []  # {"id", "input", "expected", "category" (lab12.CATEGORIES), "severity"}
scorers = []  # {"name", "type" (lab12.SCORER_TYPES), "fn": fn(expected, actual) -> bool}
transforms = []  # {"name", "fn": fn(text) -> text}  (must not change the correct extraction)
traceability = {}  # requirement id -> [golden ids]
validation_scenarios = []  # {"moe", "scenario", "users", "measure"}


def run_suite(system):
    """Run your suite against a system(text) -> claims. Return {"passed": bool, "failures": [...]}."""
    raise NotImplementedError

## 7. Public validation

In [ ]:
submission = {"golden_set": golden_set, "scorers": scorers, "transforms": transforms, "traceability": traceability, "validation_scenarios": validation_scenarios, "run_suite": run_suite}
result = fmse.check_public(LAB_ID, submission)
result.show()

## 8. Robustness / adversarial probes

Extra adversarial conditions beyond the graded suite. They do not change your score; they show where your solution is brittle.

In [ ]:
probe_results = fmse.probe(LAB_ID, submission)

## 9. Engineering reflection

Graded in the portal and stored with your artifact. Draft it here if useful:
- **Decision:** Which part of your suite caught the most mutants, and why that part?
- **Trade-offs:** What does your suite cost to run, and which checks would you drop under a time budget?
- **Remaining risks:** What kind of fault would your suite still miss?

In [ ]:
reflection_draft = {"decision": "", "tradeoffs": "", "remaining_risks": ""}

## 10. Export artifact / completion result

Exports the **Evaluation Plan + Golden Dataset + Test Harness** artifact and prints a completion record. Paste the record into the Module 12 page on agentic-ai.es (**Import lab result**). The record is a learning-workflow document, not a signed certificate.

In [ ]:
artifact = fmse.export_artifact(LAB_ID, {"golden_set": golden_set, "scorers": [s.get("name") for s in scorers], "transforms": [t.get("name") for t in transforms],
                            "traceability": traceability, "validation_scenarios": validation_scenarios}, title="Evaluation Plan + Golden Dataset + Test Harness", fmt="json")
record = fmse.make_completion_record(
    LAB_ID,
    result,
    {"reflection_draft": reflection_draft, **{}},
    artifact=artifact,
    guided=guided,
    probes=probe_results,
)
fmse.show_record(record)